# EfficientNet-B0：第二版 GPU 训练

点击「全部运行」。自动下载原始资源及官方 ImageNet 权重，先训练分类头 5 轮，再完整微调 25 轮。
原来的 MobileNet 94.62% 基线继续保留，最终按同一验证集的正确数和交叉熵选择推荐模型。
运行结束自动下载结果包，包括每轮预测、最后一轮权重和推荐提交包。不会自动提交学校作业。


## 1. 检查 GPU
无需为了本实验更换 Colab 已有的 CUDA/PyTorch 安装。

In [ ]:
import sys
import torch
import torchvision
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, 'Torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), '没有检测到 CUDA GPU，请先在 Colab 运行时设置中选择 GPU。'
print('GPU:', torch.cuda.get_device_name(0))
print('显存 GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


## 2. 自动准备数据、基线与新版代码
使用相同的 2340 张训练图及 130 张验证图，不添加外部训练图片。


In [ ]:
from pathlib import Path
import hashlib, json, zipfile, stat, time, urllib.request
from concurrent.futures import ThreadPoolExecutor

BASE = 'https://raw.githubusercontent.com/greatzhu-cyber/fuck-ddl/401bb1165579c3e4d3eb1bf49a054593fa3be55c/garbage-colab/assets'
EXPECTED_SHA256 = '8d13ed892ace3ee4e979da67a298317394b9c88c45d02093547f8d5f4ed2d32a'
archive = Path('/content/garbage_colab_gpu_bundle.zip')
if not archive.exists() or hashlib.sha256(archive.read_bytes()).hexdigest() != EXPECTED_SHA256:
    with urllib.request.urlopen(BASE + '/download_manifest.json', timeout=60) as response:
        download_manifest = json.load(response)
    assert download_manifest['sha256'] == EXPECTED_SHA256, '资源版本不匹配'
    cache = Path('/content/garbage_colab_download_parts')
    cache.mkdir(exist_ok=True)
    def download_part(item):
        assert Path(item['name']).name == item['name']
        target = cache / item['name']
        if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == item['sha256']:
            return target
        for attempt in range(3):
            try:
                with urllib.request.urlopen(BASE + '/' + item['name'], timeout=60) as response:
                    content = response.read()
                assert len(content) == item['size']
                assert hashlib.sha256(content).hexdigest() == item['sha256']
                target.write_bytes(content)
                return target
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(attempt + 1)
    print('正在自动下载数据和权重……', flush=True)
    with ThreadPoolExecutor(max_workers=4) as pool:
        downloaded = list(pool.map(download_part, download_manifest['parts']))
    with archive.open('wb') as output:
        for part in downloaded:
            output.write(part.read_bytes())
assert hashlib.sha256(archive.read_bytes()).hexdigest() == EXPECTED_SHA256
WORK = Path('/content/garbage_colab_gpu')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target = (WORK / item.filename).resolve()
        assert target.is_relative_to(WORK.resolve()), '无效压缩路径'
        assert not stat.S_ISLNK(item.external_attr >> 16), '不接受符号链接'
    z.extractall(WORK)
manifest = json.loads((WORK / 'bundle_manifest.json').read_text())
for name, digest in manifest.items():
    assert hashlib.sha256((WORK / name).read_bytes()).hexdigest() == digest, name
print('数据和权重自动准备完成，无需手动上传。')

# 下载并校验新版脚本。
CODE_BASE = 'https://raw.githubusercontent.com/greatzhu-cyber/fuck-ddl/master/garbage-colab/efficientnet_v2'
CODE_HASHES = {'efficientnet_model.py': 'c191cc0a34b30dd6eec6e401a6c7e2384c45d1ddf481d99e582b8905f0052666', 'efficientnet_main.py': '7ca3e9e2b25df98602b9a10ac49cfde3b7052f9d568119071583fd354599e658', 'train_efficientnet.py': '5262df45d5ea689307bcd45fbfd2d81e28835f5c19afb63eecf97eafc6faaf1b'}
for name, expected in CODE_HASHES.items():
    with urllib.request.urlopen(CODE_BASE + '/' + name, timeout=60) as response:
        content = response.read()
    assert hashlib.sha256(content).hexdigest() == expected, '代码版本不匹配，请重新打开最新 Colab 链接。'
    (WORK / name).write_bytes(content)
print('EfficientNet-B0 新版代码准备完成。')


## 3. 训练：分类头 5 轮 + 完整微调 25 轮
训练过程中没有第 8 轮早停。默认 AMP，BatchNorm 运行统计固定，标签平滑为 0。
结果默认保存在 Colab 临时磁盘；可把 SAVE_TO_DRIVE 改成 True 保存到 Google Drive。


In [ ]:
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 25
BATCH_SIZE = 32
SAVE_TO_DRIVE = False

import datetime, subprocess
run_tag = datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/efficientnet_v2_runs') / run_tag
else:
    OUTPUT = Path('/content/efficientnet_v2_runs') / run_tag
OUTPUT.mkdir(parents=True, exist_ok=False)
command = [sys.executable, str(WORK / 'train_efficientnet.py'), '--device', 'cuda',
           '--output', str(OUTPUT), '--head-epochs', str(HEAD_EPOCHS),
           '--finetune-epochs', str(FINETUNE_EPOCHS), '--batch-size', str(BATCH_SIZE)]
print('输出目录:', OUTPUT)
with (OUTPUT / 'training.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    returncode = process.wait()
assert returncode == 0, f'运行中断，请检查 {OUTPUT / "training.log"}'


## 4. 查看本次真实成绩
验证集用于选择权重，这不是学校隐藏测试成绩。原模型与新模型均保留，不预填新成绩。


In [ ]:
metrics = json.loads((OUTPUT / 'metrics.json').read_text())
assert metrics['device'] == 'cuda' and not metrics['smoke_test']
base = metrics['mobilenet_baseline']
best = metrics['efficientnet_best']
last = metrics['efficientnet_last']
print('GPU:', metrics['device_name'])
print(f"MobileNet 基线：{base['correct']}/130 = {base['accuracy']:.2%}")
print(f"EfficientNet 最佳：{best['correct']}/130 = {best['accuracy']:.2%}")
print(f"EfficientNet 最后一轮：{last['correct']}/130 = {last['accuracy']:.2%}")
print('最佳轮次：', metrics['best_epoch'], '完成轮数：', metrics['completed_epochs'])
print('修复旧错例：', metrics['fixed_baseline_errors'], '新增错例：', metrics['introduced_new_errors'])
print('推荐模型：', metrics['recommended_model'])
print('保存权重 CPU 重载检查：', metrics['saved_cpu_check'])
print('耗时（分钟）：', round(metrics['elapsed_seconds']/60, 2))


## 5. 下载结果
包含每轮逐图预测、最佳候选与最后一轮权重。推荐提交包使用 PyTorch，仍需学校平台测试。


In [ ]:
from google.colab import files
with zipfile.ZipFile(OUTPUT / 'efficientnet_v2_results.zip', 'a', zipfile.ZIP_DEFLATED) as z:
    if 'training.log' not in z.namelist():
        z.write(OUTPUT / 'training.log', 'training.log')
files.download(str(OUTPUT / 'efficientnet_v2_results.zip'))


训练结束后，把下载的 efficientnet_v2_results.zip 发回当前对话，即可核对新版成绩及错例变化。
官方预训练权重约 21 MB，首次运行会自动下载。显存不足时将 BATCH_SIZE 改为 16。
